#### 環境檢查

In [1]:
!nvidia-smi

Sat Aug 22 09:13:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


### clone Github repo

In [3]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
!git clone https://{GITHUB_TOKEN}@github.com/Zhanzii9624/4GB-VRAM-RAG.git
%cd 4GB-VRAM-RAG

Cloning into '4GB-VRAM-RAG'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 42 (delta 9), reused 39 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 34.43 KiB | 641.00 KiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/4GB-VRAM-RAG


### uv建立環境

In [4]:
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.local/bin:" + os.environ['PATH']

!uv --version

uv 0.12.5 (x86_64-unknown-linux-gnu)


In [5]:
!uv sync

Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 103 packages in 27.97s
Prepared 97 packages in 1m 52s
Installed 97 packages in 803ms
 + annotated-doc==0.0.5
 + anyio==4.14.2
 + aorus-rag==0.1.0 (from file:///content/4GB-VRAM-RAG)
 + asttokens==3.0.2
 + beautifulsoup4==4.15.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.4.2
 + comm==0.2.3
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.6.1
 + cuda-toolkit==13.0.3.0
 + debugpy==1.8.21
 + diskcache==5.6.3
 + executing==2.2.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + gdown==6.1.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.28.0
 + idna==3.19
 + ipykernel==7.3.0
 + ipython==9.16.1
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 + jinja2==3.1.6
 + joblib==1.5.3
 + jupyter-client==8.9.1
 + jupyter-core==5.9.1
 + llama-cpp-python==0.3.35
 + lxml==6.1.2
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + matplotlib-inline==0.2.

In [6]:
# uv sync一次裝好pyproject.toml
!uv sync

Resolved 103 packages in 1ms
Checked 97 packages in 1ms
llama_cpp ok: 0.3.35
gdown ok: 6.1.0


In [20]:
# 確認llama-cpp-python(CUDA) & gdown
!uv run python -c "import llama_cpp; print('llama_cpp ok:', llama_cpp.__version__)"
!uv run python -c "import gdown; print('gdown ok:', gdown.__version__)"

llama_cpp ok: 0.3.35
gdown ok: 6.1.0


### 建立資料 parser, chunker, embedding

In [14]:
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [7]:
!rm -f data/processed/*.json data/embeddings/*.npy
!uv run python rag/parser.py
!uv run python rag/chunker.py
!uv run python rag/embedding.py

Saved 21 records -> /content/4GB-VRAM-RAG/data/processed/specs.json
[chunker] Built 21 chunks.
[chunker] Saved 21 chunks → /content/4GB-VRAM-RAG/data/processed/chunks.json

--- Chunk 0 ---
[類別: 作業系統] [Variant: 全部型號 (BZH / BYH / BXH)]
作業系統 (Operating System): Windows 11 Pro / Windows 11 Home / UEFI Shell OS
本規格適用於全部型號 (BZH / BYH / BXH)。

--- Chunk 1 ---
[類別: 處理器] [Variant: 全部型號 (BZH / BYH / BXH)]
處理器 (CPU / Processor): Intel Core Ultra 9 Processor 275HX (36MB cache, up to 5.4GHz, 24 cores, 24 threads)
本規格適用於全部型號 (BZH / BYH / BXH)。

--- Chunk 2 ---
[類別: 顯示器] [Variant: 全部型號 (BZH / BYH / BXH)]
顯示器 (Display / Screen): 16-inch 16:10 OLED WQXGA (2560x1600), 240Hz, 1ms, DCI-P3 100%, 500nits peak, 1,000,000:1 contrast; NVIDIA G-SYNC; NVIDIA Advanced Optimus; VESA DisplayHDR True Black 500; VESA ClearMR 10000; Pantone Validated; TÜV Rheinland Low Blue Light; Dolby Vision
本規格適用於全部型號 (BZH / BYH / BXH)。
[embedding] Loading model: intfloat/multilingual-e5-small on cpu














/content/4GB-VRAM

### 下載模型

In [8]:
import gdown
from pathlib import Path

ROOT = Path.cwd()
LOCAL_MODEL = ROOT / "models" / "Qwen2.5-3B-Instruct-Q4_K_M.gguf"
LOCAL_MODEL.parent.mkdir(exist_ok=True)
print(f"LOCAL_MODEL: {LOCAL_MODEL}")
GDRIVE_FILE_ID = "1GUJbOSy6tXiblmSbYzOuVUwaBjRh3PEW"  # downloaded from huggingface, stored in drive

if LOCAL_MODEL.exists():
    print(f"already here: {LOCAL_MODEL.stat().st_size / 1e9:.2f} GB")
else:
    print("downloading from Google Drive...")
    gdown.download(id=GDRIVE_FILE_ID, output=str(LOCAL_MODEL), quiet=False)
    print(f"done: {LOCAL_MODEL.stat().st_size / 1e9:.2f} GB")

LOCAL_MODEL: /content/4GB-VRAM-RAG/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf
downloading from Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1GUJbOSy6tXiblmSbYzOuVUwaBjRh3PEW
From (redirected): https://drive.google.com/uc?id=1GUJbOSy6tXiblmSbYzOuVUwaBjRh3PEW&confirm=t&uuid=e5cb04d0-d3b7-4b77-90d4-c930a7d1efec
To: /content/4GB-VRAM-RAG/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf
100%|██████████| 2.10G/2.10G [00:27<00:00, 77.3MB/s]

done: 2.10 GB


### VRAM測量

In [9]:
# 單位MB
# !nvidia-smi --query-gpu=memory.used,memory.free,memory.total --format=csv,noheader,nounits

0, 14913, 15360


### Load LLM + Streaming測試 + TTFT/TPS測量

In [22]:
%%writefile scripts/demo_streaming.py
"""載入模型並跑一次 streaming 測試，量測 TTFT / TPS / VRAM。"""
import time
import subprocess
from inference.llama_engine import LlamaEngine
from rag.prompt import build_prompt

def get_vram_used_mb():
    out = subprocess.check_output([
        "nvidia-smi", "--query-gpu=memory.used",
        "--format=csv,noheader,nounits"
    ])
    return int(out.decode().strip())

print(f"VRAM before loading model: {get_vram_used_mb()} MB")

engine = LlamaEngine(n_gpu_layers=-1, n_ctx=2048, verbose=False)

print(f"VRAM after loading model:  {get_vram_used_mb()} MB")

dummy_ctx = [
    "[類別: 顯示晶片] [Variant: BZH]\n顯示晶片 (GPU / Graphics): NVIDIA GeForce RTX 5090 Laptop GPU, 24GB GDDR7, 175W Maximum Graphics Power with Dynamic Boost\n產品型號 BZH 專屬規格。"
]
prompt = build_prompt("BZH 的顯示晶片規格是什麼？", dummy_ctx)

start = time.perf_counter()
first_token_time = None
n_tokens = 0

for tok in engine.stream(prompt, max_new_tokens=150):
    if first_token_time is None:
        first_token_time = time.perf_counter()
    print(tok, end="", flush=True)
    n_tokens += 1

end = time.perf_counter()
print()
print(f"VRAM after generation:     {get_vram_used_mb()} MB")
print(f"TTFT: {(first_token_time - start) * 1000:.1f} ms")
print(f"TPS: {n_tokens / (end - first_token_time):.2f} tokens/sec")

Overwriting scripts/demo_streaming.py


In [23]:
!uv run python scripts/demo_streaming.py

VRAM before loading model: 0 MB
[llama_engine] Loading model: Qwen2.5-3B-Instruct-Q4_K_M.gguf
[llama_engine] n_gpu_layers=-1, n_ctx=2048
[llama_engine] Model ready.
VRAM after loading model:  2315 MB
BZH 的顯示晶片規格如下：

- GPU 名稱：NVIDIA GeForce RTX 5090 Laptop GPU
- VRAM 容量：24GB GDDR7
- 最大功耗：175W
VRAM after generation:     2349 MB
TTFT: 357.7 ms
TPS: 64.99 tokens/sec


### Retriever 快速驗證

In [15]:
%%writefile scripts/demo_retriever.py
"""驗證 chunker + embedding + retriever 是否能正常串起來。"""
from rag.chunker import load_chunks
from rag.embedding import Embedder
from rag.retriever import build_retriever

chunks = load_chunks()
embedder = Embedder(device="cpu")
embeddings = embedder.load_embeddings()
retriever = build_retriever(chunks, embeddings, embedder)

print(f"loaded {len(chunks)} chunks, retriever ready: {retriever is not None}")

# 簡單測試一次檢索
query = "BZH 的顯示晶片規格是什麼？"
results = retriever.search(query, top_k=3) if hasattr(retriever, "search") else None
if results is not None:
    for r in results:
        print("-", r)

Overwriting scripts/demo_retriever.py


In [16]:
!uv run python scripts/demo_retriever.py

[embedding] Loading model: intfloat/multilingual-e5-small on cpu

/content/4GB-VRAM-RAG/rag/embedding.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"[embedding] Model loaded. Embedding dim: {self.model.get_sentence_embedding_dimension()}")
[embedding] Model loaded. Embedding dim: 384
[embedding] Loaded embeddings shape: (21, 384)
loaded 21 chunks, retriever ready: True


#### 處理中文斷詞

In [24]:
# 加入正式依賴，寫進 pyproject.toml，之後 uv sync 都會裝好
!uv add jieba

Resolved 104 packages in 1.62s
Prepared 2 packages in 2.58s
Uninstalled 1 package in 0.38ms
Installed 2 packages in 1ms
 ~ aorus-rag==0.1.0 (from file:///content/4GB-VRAM-RAG)
 + jieba==0.42.1


### vector+keyword fusion

In [25]:
%%writefile rag/hybrid_retriever.py
"""
手寫 Hybrid Retrieval：vector search (cosine similarity) + keyword/exact match，
兩者分數各自做 min-max normalize 後加權融合。
中文查詢用 jieba 斷詞處理，避免整句字串比對抓不到關鍵字的問題。
"""
import re
import numpy as np
import jieba


def _tokenize(text: str) -> list[str]:
    """中英混合斷詞：中文用 jieba 斷詞，英文/數字用簡單 regex 切出 token。"""
    text = text.lower()
    tokens = []
    for seg in jieba.cut(text):
        seg = seg.strip()
        if not seg:
            continue
        # jieba 對英文/數字通常會整段切出來，這裡再切一次確保英文字/數字分開
        for sub in re.findall(r"[a-z0-9]+|[\u4e00-\u9fff]", seg):
            tokens.append(sub)
    return tokens


def _keyword_score(query_tokens: list[str], chunk_tokens: list[str]) -> float:
    """簡單的關鍵字重疊分數：query token 命中 chunk 的比例（類似 recall）。"""
    if not query_tokens:
        return 0.0
    chunk_set = set(chunk_tokens)
    hits = sum(1 for t in query_tokens if t in chunk_set)
    return hits / len(query_tokens)


def _minmax_normalize(scores: np.ndarray) -> np.ndarray:
    lo, hi = scores.min(), scores.max()
    if hi - lo < 1e-9:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)


class HybridRetriever:
    def __init__(self, chunks, embeddings, embedder, alpha: float = 0.6):
        """
        alpha: vector score 的權重，(1 - alpha) 是 keyword score 的權重。
        alpha 越高越吃語意相似度，越低越吃關鍵字精確命中（適合處理型號、規格數字這種精確查詢）。
        """
        self.chunks = chunks
        self.embeddings = embeddings  # shape: (N, dim)，需先 L2 normalize
        self.embedder = embedder
        self.alpha = alpha
        self.chunk_tokens = [_tokenize(c["text"] if isinstance(c, dict) else str(c)) for c in chunks]

    def search(self, query: str, top_k: int = 3):
        # --- vector score ---
        query_vec = self.embedder.model.encode([query], normalize_embeddings=True)[0]
        vector_scores = self.embeddings @ query_vec  # cosine similarity（假設 embeddings 已 normalize）

        # --- keyword score ---
        query_tokens = _tokenize(query)
        keyword_scores = np.array([
            _keyword_score(query_tokens, ct) for ct in self.chunk_tokens
        ])

        # --- fusion ---
        v_norm = _minmax_normalize(vector_scores)
        k_norm = _minmax_normalize(keyword_scores)
        final_scores = self.alpha * v_norm + (1 - self.alpha) * k_norm

        top_idx = np.argsort(-final_scores)[:top_k]
        results = []
        for i in top_idx:
            results.append({
                "chunk": self.chunks[i],
                "vector_score": float(vector_scores[i]),
                "keyword_score": float(keyword_scores[i]),
                "final_score": float(final_scores[i]),
            })
        return results


def build_hybrid_retriever(chunks, embeddings, embedder, alpha: float = 0.6):
    return HybridRetriever(chunks, embeddings, embedder, alpha=alpha)

Writing rag/hybrid_retriever.py


### Prompt拒答原則

In [26]:
%%writefile rag/prompt.py
"""手動組 system prompt + context + 問題，支援繁中/英文混合。"""

SYSTEM_PROMPT = """你是一個專業的筆電規格問答助手，只能根據提供的「參考資料」回答問題。

規則：
1. 只能使用「參考資料」裡的內容回答，不可以憑常識或記憶猜測、編造任何規格數字。
2. 如果參考資料裡找不到能回答這個問題的資訊，必須明確回答「抱歉，目前資料中找不到相關規格資訊」，不可以硬湊答案。
3. 使用者可能用繁體中文、英文，或中英混合提問，請用使用者提問的語言風格回答（中文為主時用中文回答，英文為主時可用英文回答）。
4. 回答時盡量簡潔、條列式列出規格重點，避免多餘的閒聊。
"""


def build_prompt(question: str, context_chunks: list[str]) -> str:
    if context_chunks:
        context_text = "\n\n".join(f"[參考資料 {i+1}]\n{c}" for i, c in enumerate(context_chunks))
    else:
        context_text = "（無檢索到任何相關參考資料）"

    return f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
參考資料：
{context_text}

問題：{question}<|im_end|>
<|im_start|>assistant
"""

Overwriting rag/prompt.py


### 測試TTFT/TPS + 10-15 題定性測試

In [34]:
%%writefile scripts/eval_benchmark.py
"""
系統評測：
1. 拆解量測 embedding 延遲 / retrieval 延遲 / LLM prefill 延遲 / TPS，每題跑 N 次取平均
2. 定性測試集：中英混合、跨 variant 比較、拒答測試
"""
import time
import json
import statistics
from pathlib import Path

from rag.chunker import load_chunks
from rag.embedding import Embedder
from rag.hybrid_retriever import build_hybrid_retriever
from rag.prompt import build_prompt
from inference.llama_engine import LlamaEngine

N_RUNS = 3
TOP_K = 6  # 從 3 調成 6：21 筆資料量小，多帶一點 context 對跨 variant 比較很重要

TEST_SET = [
    {"id": "q1", "type": "中文-一般", "question": "BZH 的顯示晶片規格是什麼？"},
    {"id": "q2", "type": "英文-一般", "question": "What is the GPU of the BZH variant?"},
    {"id": "q3", "type": "中英混合", "question": "請問 BZH 這台 laptop 的 VRAM capacity 是多少？"},
    {"id": "q4", "type": "跨variant比較", "question": "BZH、BYH、BXH 三個型號中，哪一個顯示晶片的最大功耗最高？"},
    {"id": "q5", "type": "跨variant比較", "question": "BYH 的顯示晶片 VRAM 是多少？跟 BZH 差多少？"},
    {"id": "q6", "type": "拒答測試", "question": "這台筆電螢幕支援觸控功能嗎？"},  # 規格表沒有觸控欄位
    {"id": "q7", "type": "拒答測試-無關問題", "question": "今天天氣如何？"},
    {"id": "q8", "type": "拒答測試", "question": "這台筆電的保固期限是幾年？"},  # 規格表沒有保固欄位
    {"id": "q9", "type": "精確數字查詢", "question": "175W 對應的是哪些型號？"},  # 注意：BZH 和 BYH 都是 175W
    {"id": "q10", "type": "一般查詢-英文", "question": "How much RAM (system memory) can this laptop support?"},
    {"id": "q11", "type": "一般查詢-中文", "question": "電池容量是多少？"},
    {"id": "q12", "type": "中英混合", "question": "這台 laptop 的 keyboard 有支援 RGB 嗎？"},
    {"id": "q13", "type": "中英混合", "question": "連接埠 right side規格有什麼"},
    {"id": "q14", "type": "中英混合", "question": "BZH、BYH、BXH差在哪? What's the difference?"},
    {"id": "q15", "type": "中英混合", "question": "BYH有NVIDIA GeForce RTX 5070 Ti和GPU16GB對嗎? what else special?"},
]


def timed(fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = (time.perf_counter() - start) * 1000
    return result, elapsed


def run_single(question: str, embedder, retriever, engine):
    _, embed_ms = timed(embedder.model.encode, [question], normalize_embeddings=True)

    results, retrieval_ms = timed(retriever.search, question, TOP_K)
    # 這裡確保 chunk_text 是字串
    context_chunks = [r["chunk"]["text"] if isinstance(r["chunk"], dict) else str(r["chunk"]) for r in results]

    prompt = build_prompt(question, context_chunks)

    start = time.perf_counter()
    first_token_time = None
    n_tokens = 0
    answer_tokens = []
    for tok in engine.stream(prompt, max_new_tokens=200):
        if first_token_time is None:
            first_token_time = time.perf_counter()
        answer_tokens.append(tok)
        n_tokens += 1
    end = time.perf_counter()

    prefill_ms = (first_token_time - start) * 1000
    tps = n_tokens / (end - first_token_time) if first_token_time else 0.0

    return {
        "embed_ms": embed_ms,
        "retrieval_ms": retrieval_ms,
        "prefill_ms": prefill_ms,
        "tps": tps,
        "n_tokens": n_tokens,
        "answer": "".join(answer_tokens),
        # 存檔前轉成純字串列表，避免 JSON serializable 錯誤
        "retrieved": context_chunks,
    }


def main():
    chunks = load_chunks()
    embedder = Embedder(device="cpu")
    embeddings = embedder.load_embeddings()
    retriever = build_hybrid_retriever(chunks, embeddings, embedder)
    engine = LlamaEngine(n_gpu_layers=-1, n_ctx=2048, verbose=False)

    all_results = []
    for case in TEST_SET:
        print(f"\n=== [{case['id']}] {case['type']}: {case['question']} ===")
        runs = [run_single(case["question"], embedder, retriever, engine) for _ in range(N_RUNS)]

        avg = {
            "embed_ms": statistics.mean(r["embed_ms"] for r in runs),
            "retrieval_ms": statistics.mean(r["retrieval_ms"] for r in runs),
            "prefill_ms": statistics.mean(r["prefill_ms"] for r in runs),
            "tps": statistics.mean(r["tps"] for r in runs),
        }
        print(f"embedding: {avg['embed_ms']:.1f} ms | retrieval: {avg['retrieval_ms']:.1f} ms "
              f"| prefill(TTFT): {avg['prefill_ms']:.1f} ms | TPS: {avg['tps']:.2f}")
        print(f"回答: {runs[-1]['answer'][:200]}")
        print(f"檢索到 {len(runs[-1]['retrieved'])} 筆 chunk")

        all_results.append({
            "id": case["id"],
            "type": case["type"],
            "question": case["question"],
            "avg_embed_ms": avg["avg_embed_ms" if "avg_embed_ms" in avg else "embed_ms"],
            "avg_retrieval_ms": avg["retrieval_ms"],
            "avg_prefill_ms": avg["prefill_ms"],
            "avg_tps": avg["tps"],
            "sample_answer": runs[-1]["answer"],
            "retrieved_chunks": runs[-1]["retrieved"],
        })

    out_path = Path("eval_results.json")
    out_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n結果已存到 {out_path}")


if __name__ == "__main__":
    main()

Overwriting scripts/eval_benchmark.py


In [35]:
!uv run python scripts/eval_benchmark.py

[embedding] Loading model: intfloat/multilingual-e5-small on cpu

/content/4GB-VRAM-RAG/rag/embedding.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"[embedding] Model loaded. Embedding dim: {self.model.get_sentence_embedding_dimension()}")
[embedding] Model loaded. Embedding dim: 384
[embedding] Loaded embeddings shape: (21, 384)
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.661 seconds.
Prefix dict has been built successfully.
[llama_engine] Loading model: Qwen2.5-3B-Instruct-Q4_K_M.gguf
[llama_engine] n_gpu_layers=-1, n_ctx=2048
[llama_engine] Model ready.

=== [q1] 中文-一般: BZH 的顯示晶片規格是什麼？ ===
embedding: 40.4 ms | retrieval: 36.4 ms | prefill(TTFT): 387.5 ms | TPS: 61.67
回答: BZH 的顯示晶片規格是 NVIDIA GeForce RTX 5090 Laptop GPU，具有 24GB GDDR7，最大_graphics 功率為 175W，支援動態增強。
檢索到 6 筆 chunk

=== [q2] 英文-一般: What is the GPU of the BZH variant? ===
embed